# 제품 사전 (규칙 버전)

DART와 뉴스에 나온 **제품·원자재·브랜드 이름을 같은 것끼리 묶어 제품 노드 하나로** 만든다(`graphDB.md` 5.1 제품 계층, 6장). 국내 기업과 해외 기업은 직접 관계가 없어도 같은 제품 노드를 거쳐 이어진다(SK하이닉스 → HBM ← Micron).

- **규칙 버전**이다(프로토타입). 영어 이름 정규화와 검색어로 묶고, 계층은 추출 때 붙인 한 단계 위 범주(`category`)만 쓴다. 실제 v5 데이터로 할 때는 LLM으로 같은 제품 묶기, 제품군 > 제품 > 세부 제품 계층, 후보 하위 제품(원문에 없는 FC-BGA 등)을 만든다(6장).
- **제품·원자재 사전과 브랜드 사전을 나눈다.** 이름이 같아도 다른 것을 가리키는 경우가 있다(DART의 "Steam"은 증기, 뉴스의 "Steam"은 Valve의 게임 플랫폼).
- 서버나 LLM에 요청하지 않는다.

**입력**

| 파일 | 쓰는 것 |
|---|---|
| `source/dart/business/final/dart_business_v5.json` | 제품(이름, 영어 이름, 범주, 검색어, 용도, 단계), 원재료(품목, 영어 이름), 브랜드(이름, 영어 이름, 소유) |
| `source/news/final/news_final.json` | 뉴스 개체 중 제품·기술, 원자재·지표(→ 제품 사전), 상품·브랜드(→ 브랜드 사전) |
| `source/dart/parsed/companies.csv` | 회사의 WI26 업종(제품이 어느 업종에 쓰이는지) |

**출력** (`source/graph/`)

| 파일 | 한 줄 |
|---|---|
| `products.csv` | 제품·원자재 노드 하나: ID, 대표 영어·한글 이름, 범주, 검색어, 만드는 국내 상장사 수, 뉴스 기사 수, 관련 업종, 일반적인 제품인지 |
| `brands.csv` | 브랜드 노드 하나 |
| `product_mentions.csv` | 언급 하나: 출처, 문서, 자리, 원래 이름, 붙은 ID |
| `categories.csv` | 범주 하나: 제품 수, 관련 업종 |

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `name_unifier` → **`product_dictionary`** → `product_llm`(LLM 실행) → `product_apply`(LLM 사전 적용) → `graph_db` |
| 외부 요청·비용 | 없음(규칙 버전). 처음 한 번만 영어 사전 자료(WordNet, 약 10MB)를 `source/nltk_data/`로 받는다. LLM 버전은 `product_llm`·`product_apply`가 맡는다 |
| 다시 실행 | 언제든 된다. 결과 파일은 내용이 바뀔 때만 다시 쓴다 |
| 처리 순서 | 이름 정규화 → 언급 모으기 → 같은 제품끼리 묶기(제품·브랜드 따로) → 노드 만들기 → 범주 표 → 저장 → 확인 |

## 1. 설정

In [1]:
# ── 설정: 폴더 ──
import collections
import functools
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
OUT = ROOT / 'source' / 'graph'
OUT.mkdir(parents=True, exist_ok=True)
HANGUL = re.compile('[가-힣]')

# 영어 사전(WordNet): 단수형을 정할 때 쓴다. 프로젝트 안(source/nltk_data)에 두고, 없으면 처음 한 번 받는다(NLTK 자료 저장소, 약 10MB)
import nltk
NLTK_DIR = ROOT / 'source' / 'nltk_data'
nltk.data.path.insert(0, str(NLTK_DIR))
if not (NLTK_DIR / 'corpora' / 'wordnet.zip').exists():
    nltk.download('wordnet', download_dir=str(NLTK_DIR), quiet=True)
from nltk.corpus import wordnet as wn
assert str(NLTK_DIR) in str(nltk.data.find('corpora/wordnet.zip')), 'WordNet을 프로젝트 폴더에서 읽지 못했다'

## 2. 이름 정규화

- 소문자, 악센트 제거, `&` → and, 문장부호를 공백으로(`+`는 남긴다. 예: Wi-Fi 6E+). "U.S." 같은 점 약어는 붙여 쓴다(US).
- **대문자 약칭은 대문자로 남긴다**: 2~4자 대문자(PET, HBM, DRAM)나 숫자가 섞인 6자 이하(HBM3E, DDR5). 플라스틱 소재 "PET"와 반려동물 "pet"을 가른다. 그보다 긴 대문자 단어(FLASH, DOOSAN)는 소문자로 바꾼다.
- **단수형**: 영어 사전(WordNet)이 아는 명사면 사전의 단수형(cases → case, potatoes → potato)이고, 복수형 자체가 표제어면 그대로 둔다(series, news, logistics, Texas). 사전에 없는 단어(smartphones, chiplets)는 기본 규칙(-ies → -y, 끝의 s 떼기)을 쓴다. 예외 단어 목록은 두지 않는다.
- 끝의 "and others", "etc"는 지운다("polypropylene and others" = "polypropylene"). 끝의 "service(s)", "product(s)"도 지운다("leasing services" = "leasing").
- **브랜드 사전은 고유명사라** 단수형으로 바꾸지 않고(Pringles, Siemens), 대소문자도 가리지 않는다(KOLON SPORT = kolon sport).
- 한글만 있는 이름은 띄어쓰기·문장부호를 지운 한글 키를 쓴다.

In [2]:
# ── 이름 정규화: 같은 제품이면 같은 키가 나오게 이름을 바꾼다 ──
def singular_rule(w):
    """사전에 없는 단어의 단수형: -ies → -y, -sses·-xes·-zes·-ches·-shes는 es를 떼고, 그 밖의 끝의 s를 뗀다(-ss·-us·-is는 그대로)."""
    if w.endswith('ies'):
        return w[:-3] + 'y'
    if w.endswith(('sses', 'xes', 'zes', 'ches', 'shes')):
        return w[:-2]
    if w.endswith('s') and not w.endswith(('ss', 'us', 'is')):
        return w[:-1]
    return w


@functools.lru_cache(maxsize=None)
def singular(w):
    """영어 단어의 단수형. 3자 이하·숫자 섞인 말은 그대로.
    영어 사전(WordNet)이 아는 명사면 사전의 단수형(cases → case, potatoes → potato)이고, 복수형 자체가 표제어면 그대로 둔다
    (series, news, logistics, Texas). 사전에 없는 단어(smartphones, chiplets)는 기본 규칙을 쓴다."""
    if len(w) <= 3 or not w.isalpha():
        return w
    forms = wn._morphy(w, wn.NOUN)          # 사전에 있는 기본형 후보(없으면 빈 목록)
    if forms:
        shorter = [f for f in forms if len(f) < len(w)]
        return min(shorter, key=len) if shorter else w
    return singular_rule(w)


DOTTED = re.compile(r'\b(?:[A-Za-z]\.){2,}')              # U.S., U.K.
TAIL = re.compile(r'\s+(?:and others?(?: \w+)?|etc|and so on)$')   # "polypropylene and others" = "polypropylene"


def is_acronym(t):
    """대문자 약칭인가: 2~4자 대문자(PET, HBM, DRAM)이거나 숫자가 섞인 6자 이하 대문자(HBM3E, DDR5). 끝의 s는 복수(SMRs)."""
    core = t[:-1] if len(t) > 2 and t.endswith('s') and t[:-1].isupper() else t
    if not core.isupper() or core.isdigit():
        return False
    return (core.isalpha() and 2 <= len(core) <= 4) or (any(ch.isdigit() for ch in core) and len(core) <= 6)


def norm_p(name, book='제품'):
    """이름을 비교용으로 정규화한다(악센트·문장부호 제거, 단수형, 끝의 and others·service·product 제거).
    제품 사전은 대문자 약칭을 남겨 일반어와 가르고(PET ≠ pet), 브랜드 사전은 고유명사라 단수형으로 바꾸지 않고 대소문자도 가리지 않는다."""
    s = unicodedata.normalize('NFKD', name or '').encode('ascii', 'ignore').decode().replace('&', ' and ')
    s = DOTTED.sub(lambda m: m.group(0).replace('.', ''), s)   # "U.S." → "US"
    toks = []
    for t in re.sub(r'[^A-Za-z0-9+ ]+', ' ', s).split():
        if book == '브랜드':
            toks.append(t.lower())
        elif is_acronym(t):
            toks.append(t[:-1] if t.endswith('s') and len(t) > 2 and t[:-1].isupper() else t)
        else:
            toks.append(singular(t.lower()))
    s = TAIL.sub('', ' '.join(toks))
    toks = s.split()
    # "leasing service"와 "leasing"을 같게 본다
    while len(toks) > 1 and toks[-1] in ('service', 'product'):
        toks = toks[:-1]
    return ' '.join(toks)


def key_of(name, name_en='', book='제품'):
    """영어 이름이 있으면 en:, 없으면 원래 이름(한글이면 ko:, 아니면 en:)."""
    if name_en and not HANGUL.search(name_en) and norm_p(name_en, book):
        return 'en:' + norm_p(name_en, book)
    if name and HANGUL.search(name):
        return 'ko:' + re.sub(r'[\s.,·\-()\[\]/]', '', name).lower()
    return 'en:' + norm_p(name, book) if name and norm_p(name, book) else None


# 정규화 예시
for n, b in [('High bandwidth memory', '제품'), ('LNG carriers', '제품'), ('Leasing services', '제품'), ('Wärtsilä 46DF', '제품'),
             ('Batteries', '제품'), ('cases', '제품'), ('Diabetes drugs', '제품'), ('West Texas Intermediate', '제품'), ('Smartphones', '제품'),
             ('polypropylene and others', '제품'), ('U.S. crude oil inventories', '제품'), ('FLASH charging', '제품'), ('PET resin', '제품'),
             ('HBM3E', '제품'), ('KOLON SPORT', '브랜드'), ('Pringles', '브랜드')]:
    print(f'{n!r:32} {b} → {key_of("", n, b)}')


'High bandwidth memory'          제품 → en:high bandwidth memory
'LNG carriers'                   제품 → en:LNG carrier
'Leasing services'               제품 → en:leasing
'Wärtsilä 46DF'                  제품 → en:wartsila 46DF
'Batteries'                      제품 → en:battery
'cases'                          제품 → en:case
'Diabetes drugs'                 제품 → en:diabetes drug
'West Texas Intermediate'        제품 → en:west texas intermediate
'Smartphones'                    제품 → en:smartphone
'polypropylene and others'       제품 → en:polypropylene
'U.S. crude oil inventories'     제품 → en:US crude oil inventory
'FLASH charging'                 제품 → en:flash charging
'PET resin'                      제품 → en:PET resin
'HBM3E'                          제품 → en:HBM3E
'KOLON SPORT'                    브랜드 → en:kolon sport
'Pringles'                       브랜드 → en:pringles


## 3. 언급 모으기

| 사전 | 출처 | 자리 |
|---|---|---|
| 제품·원자재 | DART | 제품(`products`), 원재료(`materials`의 품목) |
| 제품·원자재 | 뉴스 | 개체 중 제품·기술, 원자재·지표 |
| 브랜드 | DART | 브랜드(`brands`) |
| 브랜드 | 뉴스 | 개체 중 상품·브랜드 |

In [3]:
# ── 언급 모으기: DART 제품·원재료·브랜드와 뉴스 개체를 한 표로 ──
companies = pd.read_csv(ROOT / 'source' / 'dart' / 'parsed' / 'companies.csv', dtype=str).fillna('').set_index('stock_code')
biz = json.loads((ROOT / 'source' / 'dart' / 'business' / 'final' / 'dart_business_v5.json').read_text(encoding='utf-8'))['companies']
news = json.loads((ROOT / 'source' / 'news' / 'final' / 'news_final.json').read_text(encoding='utf-8'))['articles']
rows = []


def add(book, source, doc, field, name, name_en, category, terms, sectors, ref=''):
    """언급 한 줄을 더한다. book은 제품 사전/브랜드 사전 중 어디에 넣을지."""
    rows.append(dict(book=book, source=source, doc=doc, field=field, ref=ref, name=(name or '').strip(), name_en=(name_en or '').strip(),
                     category=(category or '').strip(), terms=[t.strip() for t in terms if t and t.strip()], sectors=list(sectors)))


# DART: 제품·원재료는 제품 사전, 브랜드는 브랜드 사전. 업종은 그 회사의 WI26 업종
for code_, c in biz.items():
    sec = [companies.at[code_, 'wi26_sector']] if code_ in companies.index else []
    for p in c['products']:
        add('제품', 'dart', code_, '제품', p['name'], p['name_en'], p['category'], p['search_terms'], sec)
    for m in c['materials']:
        add('제품', 'dart', code_, '원재료', m['item'], m['name_en'], '', [], sec)
    for b in c['brands']:
        add('브랜드', 'dart', code_, '브랜드', b['name'], b['name_en'], '', [], sec)
# 뉴스: 제품·기술·원자재·지표는 제품 사전, 상품·브랜드는 브랜드 사전. 업종은 기사의 업종
for aid, o in news.items():
    for e in o['entities']:
        if e['type'] in ('제품·기술', '원자재·지표'):
            add('제품', 'news', aid, e['type'], e['name'], e['name_en'], e['category'], e['search_terms'], o['sectors'], e['id'])
        elif e['type'] == '상품·브랜드':
            add('브랜드', 'news', aid, e['type'], e['name'], e['name_en'], e['category'], e['search_terms'], o['sectors'], e['id'])

M = pd.DataFrame(rows)
# 언급마다 비교용 키(en:… 또는 ko:…)
M['key'] = [key_of(n, ne, b) for n, ne, b in zip(M['name'], M['name_en'], M['book'])]
M = M[M['key'].notna()].reset_index(drop=True)
display(M.groupby(['book', 'source', 'field']).size().rename('언급 수').reset_index())

,book,source,field,언급 수
0,브랜드,dart,브랜드,5015
1,브랜드,news,상품·브랜드,1689
2,제품,dart,원재료,5253
3,제품,dart,제품,13739
4,제품,news,원자재·지표,1105
5,제품,news,제품·기술,3030


## 4. 묶기

사전마다 따로 묶는다(제품과 브랜드는 서로 잇지 않는다).

1. **이름이 같다**: 정규화한 키가 같으면 한 노드.
1-2. **공백만 다른 이름**(공백을 지워 5자 이상): skin care = skincare.
2. **한 언급의 원래 이름과 영어 이름**: 원래 이름이 영어면(예: DART 제품 "HBM", 영어 이름 "High bandwidth memory") 둘을 잇는다. 원래 이름이 한 단어("Switch", "wafer")거나 3자 이하("S/W"는 소프트웨어도 강선도 된다)면 3)처럼 한 제품만 가리킬 때만 잇는다(Nintendo Switch와 네트워크 스위치, 반도체 웨이퍼와 웨이퍼 과자가 묶이지 않게).
3. **검색어**: 추출 지시문은 검색어에 "같은 것을 부르는 다른 표현"만 넣게 했다. 검색어가 **전체에서 한 제품만 가리킬 때만** 잇는다. 여러 제품이 같이 쓰는 검색어("memory chips")나 일반어("AI", "oil")는 잇지 않는다.
4. 묶인 덩어리가 너무 크면(키 20개 이상) 잘못 이어졌을 수 있어 표시한다.

In [4]:
# ── 같은 제품끼리 묶기(union-find): 제품 사전과 브랜드 사전을 따로 묶는다 ──
# 너무 일반적인 말: 이 말로는 서로 다른 제품을 잇지 않는다
GENERIC = {'ai', 'artificial intelligence', 'oil', 'crude', 'gas', 'electricity', 'power', 'energy', 'data center', 'inflation',
           'stock', 'tariff', 'ev', 'electric vehicle', 'battery', 'chip', 'semiconductor', 'technology', 'software', 'cloud',
           'steel', 'shipping', 'insurance', 'bank', 'memory', 'memory chip', 'equipment', 'material', 'part', 'component'}


def build(book):
    """한 사전(제품 또는 브랜드)의 키를 묶고, 묶음을 찾는 함수와 검색어로 이은 수를 돌려준다."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    D = M[M['book'] == book]
    # owners[이름 키] = 그 이름을 쓰는 제품들. 한 제품만 가리킬 때만 잇는다
    owners = collections.defaultdict(set)
    for r in D.itertuples():
        find(r.key)
        # 근거 1: 한 언급의 원래 영어 이름과 표준 영어 이름은 같은 제품
        if r.name and not HANGUL.search(r.name) and r.name_en:
            n = norm_p(r.name, book)
            k2 = 'en:' + n
            if not n or n.lower() in GENERIC or k2 == r.key:
                continue
            if len(n.split()) == 1 or len(n.replace(' ', '')) <= 3:
                owners[k2].add(r.key)        # 한 단어("Switch", "wafer")나 3자 이하("S/W") 이름은 한 제품만 가리킬 때만 잇는다(아래)
            else:
                union(r.key, k2)
    # 근거 1-2: 공백만 다른 이름(skin care = skincare, smart city = smartcity). 공백을 지워 5자 이상일 때만
    by_compact = collections.defaultdict(list)
    for k in D['key'].unique():
        if k.startswith('en:') and len(k[3:].replace(' ', '')) >= 5:
            by_compact[k[3:].replace(' ', '')].append(k)
    for ks in by_compact.values():
        for k in ks[1:]:
            union(ks[0], k)
    for r in D.itertuples():
        # 근거 2: 검색어. 여러 제품이 같이 쓰는 검색어로는 잇지 않는다
        for t in r.terms:
            k = 'en:' + norm_p(t, book)
            if norm_p(t, book) and norm_p(t, book).lower() not in GENERIC and k != r.key:
                owners[k].add(r.key)
    linked = 0
    # 이름·검색어가 전체에서 한 제품만 가리키면 잇는다
    for k, os_ in owners.items():
        roots = {find(o) for o in os_}
        if len(roots) == 1:
            union(next(iter(roots)), k)
            linked += 1
    return find, linked


find_p, linked_p = build('제품')
find_b, linked_b = build('브랜드')
M['root'] = [find_p(k) if b == '제품' else find_b(k) for b, k in zip(M['book'], M['key'])]
print(f'검색어로 이은 이름: 제품 {linked_p}, 브랜드 {linked_b}')

검색어로 이은 이름: 제품 12607, 브랜드 821


## 5. 노드 만들기

- ID: 제품은 `P:`, 브랜드는 `B:` 뒤에 덩어리에서 가장 많이 쓰인 키(영어 우선).
- 대표 이름: 가장 많이 쓰인 영어 이름과 한글 이름. 범주: 가장 많이 붙은 범주.
- 검색어: 대표 이름과 덩어리의 검색어(일반어 제외, 많이 쓰인 순 최대 5개).
- **만드는 국내 상장사 수**(`n_kr_makers`): DART에서 이 제품을 제품으로 적은 회사 수. **뉴스 기사 수**(`n_news`).
- **일반적인 제품**(`generic`): 국내 상장사 10곳 이상이 만드는 제품(예: 부동산 임대, 화장품) 또는 일반어. 검색어로는 순위를 낮춘다(5.1 "검색어는 구체적인 노드를 우선").
- **관련 업종**: 이 제품을 적은 DART 회사의 WI26 업종과 뉴스 기사의 업종(많은 순 3개).

In [5]:
# ── 노드 만들기: 묶음 하나가 노드 하나. 대표 이름, 검색어, 만드는 상장사 수 등을 셈 ──
def make_nodes(book, prefix):
    """한 사전의 묶음마다 노드 한 줄을 만든다."""
    D = M[M['book'] == book]
    out = []
    for root, g in D.groupby('root'):
        keys = collections.Counter(g['key'])
        en_keys = [(n, k) for k, n in keys.items() if k.startswith('en:')]
        # ID는 가장 많이 쓰인 영어 키(없으면 가장 많이 쓰인 키)
        main = max(en_keys)[1] if en_keys else keys.most_common(1)[0][0]
        names_en = collections.Counter(n for n in g['name_en'] if n and not HANGUL.search(n))
        names_ko = collections.Counter(n for n in g['name'] if HANGUL.search(n))
        cats = collections.Counter(c for c in g['category'] if c)
        terms = collections.Counter(t for ts in g['terms'] for t in ts if norm_p(t, book) not in GENERIC)
        secs = collections.Counter(s for ss in g['sectors'] for s in ss)
        # 이 제품을 만드는(제품·브랜드로 적은) 국내 상장사 수
        kr = g[(g['source'] == 'dart') & (g['field'].isin(['제품', '브랜드']))]['doc'].nunique()
        name_en = names_en.most_common(1)[0][0] if names_en else ''
        out.append(dict(id=prefix + main[3:], name_en=name_en, name_ko=names_ko.most_common(1)[0][0] if names_ko else '',
                        category=cats.most_common(1)[0][0] if cats else '',
                        search_terms=' | '.join(t for t, _ in terms.most_common(5)),
                        n_kr_makers=kr, n_dart_material=int((g['field'] == '원재료').sum()), n_news=g[g['source'] == 'news']['doc'].nunique(),
                        n_mentions=len(g), n_keys=len(keys), sectors=' | '.join(s for s, _ in secs.most_common(3)),
                        # 국내 상장사 10곳 이상이 만들거나 이름이 일반어면 일반적인 제품(검색어 순위를 낮춘다)
                        generic=kr >= 10 or norm_p(name_en, book) in GENERIC, root=root))
    return pd.DataFrame(out).sort_values('n_mentions', ascending=False)


P = make_nodes('제품', 'P:')
B = make_nodes('브랜드', 'B:')
# 언급마다 붙을 노드 ID
idmap = {('제품', r): i for r, i in zip(P['root'], P['id'])} | {('브랜드', r): i for r, i in zip(B['root'], B['id'])}
M['node_id'] = [idmap[(b, r)] for b, r in zip(M['book'], M['root'])]
print(f'제품·원자재 노드 {len(P):,}개(언급 {(M.book == "제품").sum():,}), 브랜드 노드 {len(B):,}개(언급 {(M.book == "브랜드").sum():,})')
print(f'뉴스와 DART에 모두 나온 제품 {((P.n_news > 0) & (P.n_mentions > P.n_news)).sum():,}개, 일반적인 제품 {P.generic.sum():,}개, 키 20개 이상인 덩어리 {(P.n_keys >= 20).sum()}개')

제품·원자재 노드 13,767개(언급 23,127), 브랜드 노드 5,596개(언급 6,704)
뉴스와 DART에 모두 나온 제품 277개, 일반적인 제품 94개, 키 20개 이상인 덩어리 0개


## 6. 범주

범주(`category`)는 추출할 때 붙인 한 단계 위 이름이다(우리말). 범주마다 제품 수와 관련 업종을 센다. 제품군 계층의 재료가 된다.

In [6]:
# ── 범주 표: 범주마다 제품 수, 뉴스 기사 수, 주 업종 ──
C = (P[P['category'] != ''].assign(sec=P['sectors'].str.split(' | ', regex=False).str[0])
     .groupby('category').agg(n_products=('id', 'size'), n_news=('n_news', 'sum'), n_kr_makers=('n_kr_makers', 'sum'),
                              top_sector=('sec', lambda s: s.value_counts().index[0] if s.notna().any() else ''))
     .sort_values('n_products', ascending=False).reset_index())
print(f'범주 {len(C):,}개')
C.head(15)

범주 5,131개


,category,n_products,n_news,n_kr_makers,top_sector
0,의약품,156,3,312,건강관리
1,자동차 부품,82,2,144,자동차
2,신약 후보물질,63,0,65,건강관리
3,비디오 게임,57,59,0,소프트웨어
4,개발 의약품,56,0,69,건강관리
5,물류 서비스,51,2,144,운송
6,가공식품,45,0,120,필수소비재
7,의류,43,0,98,"화장품,의류"
8,원료의약품,40,0,59,건강관리
9,의료기기,38,0,59,건강관리


## 7. 저장

In [7]:
# ── 저장: 제품·브랜드 노드, 언급 → 노드 대응, 범주 ──
def write_if_changed(path, df):
    """내용이 바뀌었을 때만 CSV를 쓴다(iCloud 동기화 사본이 생기지 않게)."""
    content = '﻿' + df.to_csv(index=False)
    if path.exists() and path.read_text(encoding='utf-8') == content:
        return False
    path.write_text(content, encoding='utf-8')
    return True


M['terms'] = M['terms'].map(' | '.join)
M['sectors'] = M['sectors'].map(' | '.join)
w = [write_if_changed(OUT / 'products.csv', P.drop(columns='root')),
     write_if_changed(OUT / 'brands.csv', B.drop(columns='root')),
     write_if_changed(OUT / 'product_mentions.csv', M[['book', 'source', 'doc', 'field', 'ref', 'name', 'name_en', 'category', 'node_id']]),
     write_if_changed(OUT / 'categories.csv', C)]
print('저장', dict(zip(['products', 'brands', 'product_mentions', 'categories'], w)))

저장 {'products': False, 'brands': False, 'product_mentions': False, 'categories': False}


## 8. 결과 확인

1. 뉴스와 DART를 잇는 제품(국내 상장사가 만들고 뉴스에도 나온 것)
2. 여러 이름이 하나로 묶인 예
3. 키가 많은 덩어리(잘못 묶였는지 볼 것)
4. 브랜드

In [8]:
# ── 확인 1: 뉴스와 DART를 잇는 제품 ──
print('1) 뉴스와 DART를 잇는 제품(일반적인 제품 제외, 뉴스 기사 수 순)')
display(P[(P.n_kr_makers > 0) & (P.n_news > 0) & ~P.generic].sort_values('n_news', ascending=False)
        .head(20)[['id', 'name_en', 'name_ko', 'category', 'n_kr_makers', 'n_news', 'search_terms']])

1) 뉴스와 DART를 잇는 제품(일반적인 제품 제외, 뉴스 기사 수 순)


,id,name_en,name_ko,category,n_kr_makers,n_news,search_terms
4392,P:crude oil,Crude oil,원유,원유,5,45,crude oil | petroleum | crude oil prices | petroleum crude | oil p...
7959,P:liquefied natural gas,Liquefied natural gas,LNG 등,액화천연가스,3,33,liquefied natural gas | LNG | LNG bunkering
1785,P:AI chip,AI chips,,인공지능 반도체,1,25,AI chips | artificial intelligence chips | AI accelerators | AI pr...
2727,P:energy storage system,battery energy storage systems,에너지저장장치,에너지저장장치,8,22,battery energy storage systems | ESS | BESS | energy storage syste...
4689,P:diesel fuel,Diesel fuel,경유,석유제품,9,19,diesel fuel | diesel prices | diesel | gasoil | diesel oil
6264,P:gasoline,Gasoline,휘발유,석유제품,8,17,gasoline | petrol | gasoline prices | gas prices | pump prices
6955,P:hot rolled coil,Hot-rolled coil,열연코일,열연강판,5,12,HRC | hot-rolled coil | hot-rolled steel | hot rolled coil | hot r...
11813,P:small modular reactor,small modular reactors,,원자로,1,11,small modular reactors | SMRs | SMR | small modular reactor
11446,P:semiconductor,semiconductor chips,반도체,반도체,3,10,semiconductor chips | semiconductor chip | semiconductor manufactu...
1817,P:AI agent,AI agents,,인공지능 에이전트,2,10,agentic AI | AI agent | artificial intelligence agent | agentic AI...


In [9]:
# ── 확인 2~4: 묶인 예, 키가 많은 덩어리(잘못 묶인 것이 없는지), 브랜드 ──
print('2) 여러 이름이 하나로 묶인 예')
ex = M[M.book == '제품'].groupby('node_id').agg(names=('name_en', lambda s: ' | '.join(sorted({x for x in s if x})[:6])), n=('key', 'nunique'))
display(ex[ex.n.between(3, 8)].sample(12, random_state=1))
print('3) 키가 많은 덩어리')
display(P.sort_values('n_keys', ascending=False).head(8)[['id', 'name_en', 'n_keys', 'n_mentions', 'search_terms']])
print('4) 브랜드(뉴스 기사 수 순)')
display(B.sort_values('n_news', ascending=False).head(10)[['id', 'name_en', 'category', 'n_kr_makers', 'n_news']])

2) 여러 이름이 하나로 묶인 예


,names,n
node_id,,
P:smartphone,Mobile phone | Mobile phone service | Mobile phones | Mobile servi...,4
P:steel scrap,Metal scrap | Scrap metal | Scrap steel | Scrap steel and others |...,4
P:food ingredient,Food ingredient distribution | Food ingredients | Food ingredients...,3
P:sportswear,Outdoor and sports apparel | Outdoor apparel | Sportswear | sports...,3
P:vehicle maintenance,Automated repair software | Automotive maintenance and repair | Au...,4
P:crude oil tanker,Crude oil tanker | Oil tankers | crude oil tanker | oil tanker | o...,3
P:green steel,green steel | low-carbon steel | low-emission steel,3
P:steel pipe,Steel pipe | Steel pipes | Steel tube | Welded steel pipes | steel...,3
P:security and derivative brokerage,Derivatives brokerage | Securities and derivatives brokerage | Sec...,3


3) 키가 많은 덩어리


,id,name_en,n_keys,n_mentions,search_terms
2727,P:energy storage system,battery energy storage systems,8,30,battery energy storage systems | ESS | BESS | energy storage syste...
782,P:GLP 1 receptor agonist,GLP-1 receptor agonists,5,9,GLP-1 drugs | GLP-1 receptor agonists | GLP-1 weight loss drugs | ...
1785,P:AI chip,AI chips,5,26,AI chips | artificial intelligence chips | AI accelerators | AI pr...
9822,P:peptic ulcer drug,peptic ulcer drugs,4,8,peptic ulcer drugs | peptic ulcer treatments | peptic ulcer treatm...
2469,P:vehicle maintenance,Vehicle maintenance,4,7,vehicle maintenance | automotive maintenance | automotive repair s...
5259,P:electronic component,Electronic components,4,26,automotive electronic components | electronic components | automot...
2478,P:automotive part,Automotive parts,4,25,automotive parts | auto parts | automotive components | auto compo...
12255,P:steel scrap,Steel scrap,4,22,scrap metal | ferrous scrap | metal scrap | steel scrap | scrap steel


4) 브랜드(뉴스 기사 수 순)


,id,name_en,category,n_kr_makers,n_news
4315,B:starlink,Starlink,위성 인터넷,0,21
5048,B:xbox,Xbox,게임 플랫폼,0,12
1788,B:grand theft auto vi,Grand Theft Auto VI,비디오 게임,0,12
3525,B:playstation,PlayStation,게임 플랫폼,0,9
5129,B:zepbound,Zepbound,비만 치료제,0,8
3835,B:rhode,Rhode,스킨케어 브랜드,0,8
4541,B:tesla cybertruck,Tesla Cybertruck,전기 픽업트럭,0,7
3191,B:nicotine pouches,nicotine pouches,니코틴 제품,0,7
1465,B:facebook,Facebook,소셜미디어 플랫폼,0,6
4540,B:tesla cybercab,Tesla Cybercab,로보택시,0,6
